# Avaliação do modelo fine-tunado (PubMedQA) no Google Colab

Este notebook faz o mesmo que `scripts/run_evaluate.py`: carrega o modelo fine-tunado, roda inferência no test set PubMedQA, gera **predictions.json** e calcula **Accuracy** e **Macro-F1**.

**Antes de rodar:**
1. **GPU opcional:** Para acelerar a inferência, ative GPU em *Runtime → Change runtime type → GPU*. Sem GPU funciona em CPU (mais lento).
2. **Modelo:** O adaptador PEFT deve estar em uma pasta com `adapter_config.json`, `adapter_model.safetensors` (ou `.bin`) e o tokenizer. Use a pasta onde você salvou o resultado do fine-tuning (ex.: no Drive).
3. **Dados:** É preciso ter `test.jsonl` e `test_ground_truth.json` (em `data/` se clonar o repo, ou no Drive).

**Fluxo:** Montar Drive → instalar deps → clonar/copiar projeto → configurar caminhos → rodar avaliação.

**Demora:** A inferência processa **um exemplo por vez** (sem batch). No test set completo isso pode levar de dezenas de minutos a mais de 1 h (CPU bem mais lento que GPU). Use **MAX_SAMPLES** (ex.: 50) na célula de config para um teste rápido; a barra de progresso mostra o andamento.

## 1. Montar Google Drive e instalar dependências

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q "torch>=2.0" "transformers>=4.36" "peft>=0.7" "accelerate>=0.25" "scikit-learn>=1.3" "tqdm"

## 2. Clonar o repositório (ou usar pasta no Drive)

O clone traz o código (`src/models/evaluate_pqal.py`). Se o repo não tiver `data/test.jsonl` e `data/test_ground_truth.json`, rode antes `!python scripts/run_prepare_data.py` (com `data/ori_pqal.json` em `data/`) ou use uma pasta no Drive que já tenha esses arquivos e aponte `DATA_DIR` na célula 3.

In [ ]:
!git clone https://github.com/thiagonespereira/POSTECH-FIAP-FASE3.git /content/POSTECH-FIAP-FASE3
%cd /content/POSTECH-FIAP-FASE3
print("Repositório em /content/POSTECH-FIAP-FASE3")

## 3. Configurar caminhos

Ajuste as variáveis abaixo:

- **REPO_DIR:** pasta do projeto (clone ou cópia do Drive).
- **MODEL_DIR:** pasta do **modelo fine-tunado** (adaptador PEFT + tokenizer). Se você treinou no Colab e salvou no Drive, use algo como `"/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/finetune_pqal"`.
- **DATA_DIR:** pasta que contém `test.jsonl` e `test_ground_truth.json`. Em geral é `REPO_DIR / "data"`.
- **OUT_DIR:** onde salvar `predictions.json` e `metrics.json`.
- **MAX_SAMPLES:** `None` = avaliar todos (pode demorar muito). Use um número (ex. `50`) para teste rápido e ver a barra de progresso.

In [ ]:
import sys
from pathlib import Path

# Pasta do projeto (clone ou Drive)
REPO_DIR = Path("/content/POSTECH-FIAP-FASE3")

# Pasta do modelo fine-tunado (adaptador PEFT + tokenizer)
# Ex.: no Drive após treinar: "/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/finetune_pqal"
MODEL_DIR = REPO_DIR / "outputs" / "finetune_pqal"
if not MODEL_DIR.exists():
    MODEL_DIR = Path("/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/finetune_pqal")

# Dados de teste (test.jsonl e test_ground_truth.json)
DATA_DIR = REPO_DIR / "data"

# Saída (predictions.json e metrics.json)
OUT_DIR = REPO_DIR / "outputs" / "eval"

# Limitar amostras (None = todos); use ex. 10 para teste rápido
MAX_SAMPLES = None  # ou 10, 50, etc.
MAX_NEW_TOKENS = 256

# Adicionar projeto ao path para importar src.models.evaluate_pqal
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("MODEL_DIR:", MODEL_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("MAX_SAMPLES:", MAX_SAMPLES)
print("MODEL_DIR existe?", MODEL_DIR.exists())
print("test.jsonl existe?", (DATA_DIR / "test.jsonl").exists())
print("test_ground_truth.json existe?", (DATA_DIR / "test_ground_truth.json").exists())

## 4. Rodar avaliação

Carrega o modelo (base + adaptador PEFT), roda inferência em cada exemplo do test set, extrai a decisão (yes/no/maybe) e calcula Accuracy e Macro-F1.

In [ ]:
from src.models.evaluate_pqal import evaluate

OUT_DIR.mkdir(parents=True, exist_ok=True)
predictions_path = OUT_DIR / "predictions.json"
metrics_path = OUT_DIR / "metrics.json"

n = MAX_SAMPLES if MAX_SAMPLES is not None else "todos"
print(f"Carregando modelo e rodando inferência no test set (amostras: {n})...")
metrics = evaluate(
    model_dir=MODEL_DIR,
    data_dir=DATA_DIR,
    output_predictions_path=predictions_path,
    output_metrics_path=metrics_path,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=MAX_SAMPLES,
)
print(f"Predictions salvas em: {predictions_path}")
print(f"Métricas salvas em: {metrics_path}")
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Macro-F1: {metrics['macro_f1']:.4f}")

### (Opcional) Copiar resultados para o Drive

Para não perder os arquivos ao desconectar, copie `outputs/eval/` para o Drive.

In [ ]:
# Descomente e ajuste o caminho no Drive se quiser salvar uma cópia
# import shutil
# drive_eval = Path("/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/eval")
# drive_eval.mkdir(parents=True, exist_ok=True)
# for f in ("predictions.json", "metrics.json"):
#     shutil.copy(OUT_DIR / f, drive_eval / f)
# print("Copiado para", drive_eval)